In [1]:
# 本节课学习添加情景记忆

In [2]:
profile = {
    "name": "jiarui",
    "full_name": "raojiarui",
    "user_profile_background": "Senior software engineer and also a university student"
}
prompt_instructions = {
    "triage_rules": {
        # 分类规则
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage emails.",
}
# 创建示例邮件
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
    Hi John,

    I was reviewing the API documentation for the new authentication service and noticed a few endpoints are missing. Specifically, I'm looking at:

    - /auth/refresh
    - /auth/validate

    Thanks!
    Alice
    """
}


In [3]:
# 定义长期记忆存储
from langgraph.store.memory import InMemoryStore

In [4]:
store = InMemoryStore(
    index={"embed":"ollama:qwen3-embedding:0.6b"}
)

In [5]:
email = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [6]:
data = {
    "email": email,
    "label": "respond"
}

In [7]:
import uuid

store.put(
    # 将其存入长期记忆
    ("email_assistant", "raojiarui", "examples"),
    str(uuid.uuid4()),
    data
)

In [8]:
template = """
Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content:
```
{content}
```
> Triage Result:{result}
"""

def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            template.format(
                subject=eg.value["email"]["subject"],
                to_email=eg.value["email"]["to"],
                from_email=eg.value["email"]["author"],
                content=eg.value["email"]["email_thread"][:400],
                result=eg.value["label"],
            )
        )
    return "\n\n--------------------\n\n".join(strs)

In [9]:
email_data = {
    "author": "Sarah Chen <sarah.chen@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Update: Backend API Changes Deployed to Staging",
    "email_thread": """Hi John,

wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:

- Implemented JWT refresh token rotation
- Added rate limiting for login attempts
- Updated API documentation with new endpoints

All tests are passing and the changes are ready for review. You can test it out at staging-api.company.com/auth/*

No immediate action needed from your side - just keeping you in the loop since this affects the systems you're working on.

Best regards,
Sarah
""",
}


# 查询对应邮件操作
results = store.search(
    ("email_assistant","raojiarui","examples"),
    query=str({"email": email_data}), # 查询参数
    limit=1
)

In [10]:
# 将查询得到的结果通过函数进行format
print(format_few_shot_examples(results))

Here are some previous examples:

--------------------


Email Subject: Quick question about API documentation
Email From: Alice Smith <alice.smith@company.com>
Email To: John Doe <john.doe@company.com>
Email Content:
```
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice
```
> Triage Result:respond



In [11]:
triage_system_prompt = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Background >
{user_profile_background}.
</ Background >

< Instructions >

{name} gets lots of emails. Your job is to categorize each email into one of three categories:

1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that {name} should know about but doesn't require a response
3. RESPOND - Emails that need a direct response from {name}

Classify the below email into one of these categories.

</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

There are also other things that {name} should know about, but don't require an email response. For these, you should notify {name} (using the `notify` response). Examples of this include:
{triage_notify}

Emails that are worth responding to:
{triage_email}
</ Rules >

< Few shot examples >

Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above

{examples}
</ Few shot examples >
"""

In [12]:
# 创建路由规则和创建路由
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [29]:
llm = init_chat_model("deepseek-v4-flash", extra_body={"thinking":{"type": "disabled"}})

In [30]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content"""

    reasoning : str = Field(
        # 语言模型在此生成决策
        description="Step-by-Step reasoning behind the classification."
    )

    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: ignore for irrelevant emails,notify for important information that doesn't need a response,respond for emails that need a reply",
    )

In [31]:
llm_router = llm.with_structured_output(Router) # 输出规则

In [32]:
from prompts import triage_user_prompt

In [33]:
from langgraph.graph import add_messages

class State(TypedDict):
    email_input : dict # email输入
    messages : Annotated[list, add_messages]

In [34]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

In [35]:
# 构建代理
def triage_router(state : State, config, store) -> Command[
    Literal["response_agent", "__end__"]
]:
    # load the info
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    # 添加示例：想要通过embed模型进行存储，需要添加对应存储的namespace
    namespace = (
        "email_assistant",
        config['configurable']['langgraph_user_id'],
        "examples"
    )

    examples = store.search(
        namespace,
        # 查询语句
        query=str({"email": state["email_input"]})
    )

    examples=format_few_shot_examples(examples)

    system_prompt = triage_system_prompt.format(
        full_name = profile["full_name"], #
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
        examples=examples
    )

    user_prompt = triage_user_prompt.format(
        author=author,
        to=to,
        subject=subject,
        email_thread=email_thread
    )

    result = llm_router.invoke(
        # llm_router是带有分类输出的llm，将llm的输出格式化为route
        [
            ("system", system_prompt),
            ("user", user_prompt)
            # {"role" : "system", "content": system_prompt},
            # {"role": "user", "content": user_prompt}
        ]
    )

    # 处理三种可能的结果
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        # 跳转到响应agent的节点
        update = {
            # 更新状态
            "messages": [
                # {
                #     "role": "user",
                #     "content": f"Respond to the email {state['email_input']}",
                # }
                ("user", f"Respond to email {state['email_input']}")
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        update = None
        goto = END
    elif result.classification == "notify":
        # If real life, this would do something else
        print("🔔 Classification: NOTIFY - This email contains important information")
        update = None
        goto = END
    else:
        raise ValueError(f"Invalid classification: {result.classification}")
    return Command(goto=goto, update=update)

In [36]:
from langchain_core.tools import tool


@tool  # 暂不实现任何逻辑
def write_email(to: str, subject: str, content: str) -> str:
    """Writer and send an email."""
    # Placeholder response - in real app would send email
    # 需要完成发送邮箱的邮箱地址，邮箱主题和邮箱内容
    return f"Email sent to {to} with subject '{subject}'"


@tool
def schedule_meeting(
        attendees: list[str],
        subject: str,
        duration_minutes: int,
        preferred_day: str  # 日历会议
) -> str:
    """Schedule a calendar meeting."""
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [37]:
# 创建记忆管理工具
from langmem import create_manage_memory_tool, create_search_memory_tool

In [38]:
manage_memory_tool = create_manage_memory_tool(
    namespace=(
        "email_assistant", # 命名空间值
        "{langgraph_user_id}",
        "collection"
    )
)

search_memory_tool = create_search_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)

In [39]:
# 添加新的系统提示词
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""

In [40]:
from langchain.agents.middleware import dynamic_prompt


@dynamic_prompt
def create_prompt(state):
    # 只返回字符串
    return agent_system_prompt_memory.format(
        instructions=prompt_instructions["agent_instructions"],
        **profile
    )


# 创建react代理函数
from langchain.agents import create_agent

tools = [
    write_email,
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]

response_agent = create_agent(
    "deepseek-v4-flash",
    tools=tools,
    middleware=[create_prompt],
    store=store
)

In [41]:
config = {"configurable": {"langgraph_user_id":"raojiarui"}}

In [42]:
# 创建代理
email_agent = StateGraph(State)
email_agent = email_agent.add_node(triage_router)
email_agent = email_agent.add_node("response_agent", response_agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)

In [43]:
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentaion?""",
}

In [46]:
response = email_agent.invoke(
    {"email_input" : email_input},
    config={"configurable": {"langgraph_user_id": "duwenxin"}}
)

🚫 Classification: IGNORE - This email can be safely ignored


In [49]:
data = {
    "email": {
        "author": "Tom Jones <tome.jones@bar.com>",
        "to": "John Doe <john.doe@company.com>",
        "subject": "Quick question about API documentation",
        "email_thread": """Hi John — want to buy documentation?""",
    },
    "label": "Respond"
}

In [50]:
store.put(
    ("email_assistant", "duwenxin", "examples"),
    str(uuid.uuid4()),
    data
)

In [53]:
email_input = {
    "author": "Jim Jones<jin.jones@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?????????""",
}
response = email_agent.invoke(
    {"email_input": email_input},
    config={"configurable": {"langgraph_user_id": "andrew"}}
)

🚫 Classification: IGNORE - This email can be safely ignored


In [55]:
store.list_namespaces()

[('email_assistant', 'duwenxin', 'examples'),
 ('email_assistant', 'raojiarui', 'examples')]